# Multi Document Summurizer

In [1]:
# !pip install summa pypdf torch transformers


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Part 1: Extractive Summarization
Use summa to extract top sentences from each document separately.

In [2]:
from summa import summarizer
from pypdf import PdfReader

In [3]:
# files = input("Enter the file names (space separated): ").split(" ")
files = ["one.pdf", "two.pdf", "three.pdf"]
print("Files entered:", files)

Files entered: ['one.pdf', 'two.pdf', 'three.pdf']


In [4]:
def summarize_extract(file):
    reader = PdfReader(file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    summary = summarizer.summarize(text, ratio=0.3)
    return summary

In [5]:
extractive_summaries = []
for file in files:
    summary = summarize_extract(file)
    print(f"Summary for {file}:\n{summary}\n")
    extractive_summaries.append(summary)

Summary for one.pdf:
Lifelong Program Analysis & Transformation
This paper describes LL VM (Low Level Virtual Machine),
long program analysis and transformation for arbitrary pro-
grams, by providing high-level information to compiler
transformations at compile-time, link-time, run-time, an d in
LL VM deﬁnes a common, low-level
code representation in Static Single Assignment (SSA) form ,
implement high-level language features; an instruction fo r
high-level languages (and setjmp/longjmp in C) uniformly
The LL VM compiler framework and code
transformation of programs.
compilation approach provides all these capabilities.
scribe the design of the LL VM representation and compiler
type information it provides; (b) compiler performance for
ples of the beneﬁts LL VM provides for several challenging
program analysis and transformation must be performed
Such “lifelong code
mizations performed at link-time (to preserve the beneﬁts of
separate compilation), machine-dependent optimizations at
in

Multiple definitions in dictionary at byte 0x432f for key /ToUnicode


Summary for two.pdf:
Lifelong Program Analysis & Transformation
This paper describes LL VM (Low Level Virtual Machine),
long program analysis and transformation for arbitrary pro-
grams, by providing high-level information to compiler
transformations at compile-time, link-time, run-time, an d in
LL VM deﬁnes a common, low-level
code representation in Static Single Assignment (SSA) form ,
implement high-level language features; an instruction fo r
high-level languages (and setjmp/longjmp in C) uniformly
The LL VM compiler framework and code
transformation of programs.
compilation approach provides all these capabilities.
scribe the design of the LL VM representation and compiler
type information it provides; (b) compiler performance for
ples of the beneﬁts LL VM provides for several challenging
program analysis and transformation must be performed
Such “lifelong code
mizations performed at link-time (to preserve the beneﬁts of
separate compilation), machine-dependent optimizations at
in

In [6]:
len(extractive_summaries[0])

28096

## Part 2: Abstraction summarization
In this phase the summaries extracted form each document are combined and summarised together using distilBart-12-6 model.

In [11]:
from transformers import AutoTokenizer, BartForConditionalGeneration

In [12]:
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

Loading weights: 100%|██████████| 358/358 [00:00<00:00, 19289.11it/s]


In [22]:
finalInput = ""
for extSum in extractive_summaries:
    inputs = tokenizer(extSum, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(inputs["input_ids"], num_beams=4, min_length=512, max_length=720, early_stopping=True)
    abstractive_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    finalInput += abstractive_summary + " [SEP] "
print(f"Final Abstractive Summary:\n{finalInput}\n")

Final Abstractive Summary:
 LL VM (Low Level Virtual Machine) provides high-level information to compiler transformations at compile-time, link-time or run-time . LL VM deﬁnes a common, low-level code representation in Static Single Assignment (SSA) form . It can’t guarantee type safety, memory safety, or memory safety . The LL VM representation is language-independent,for two reasons. It is complementary to high level virtual machines (e.g., Small-generation), and is not intended to be a universal compiler IR . An idle-time optimizer has not been been implemented in the language yet implemented in LL VM . An experimental LL VM system with re-resentation, including the ability to extract useful type i mation for C programs; (bmation for . C programs, for example, can extract the . type i-mation, for the purposes of the LL VM -and (c) “exception-handling instructions for implementing languag e-forms’s “properative” and “le-guided optimization between runs (“Lifelong code-based optimizat

In [ ]:
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

c:\Users\mrz\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mrz\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [21]:
inputs = tokenizer(finalInput, return_tensors="pt", max_length=4096, truncation=True)
summary_ids = model.generate(inputs["input_ids"], num_beams=4, min_length=512, max_length=6122, early_stopping=True)
finalSummary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Final Summary:\n{finalSummary}\n")

IndexError: index out of range in self